# Observation du Model

L'objectif de ce notebook est d'observer et de comprendre les mécanismes exacts d'un vrai LLM. Nous avons choisi Qwen2.5 :1.5B-Instruct comme LLM pour sa légèreté et sa modularité (série "Instruct", très performant avec un LoRA ou QLoRA).

## La classe ObserveModel

Afin d'observer le modèle de langage, nous avons créé la classe "ObserveModel" dont l'objectif est de découper étapes par étapes les calculs effectués par le modèle de langage sans adaptateur. Cette classe se trouve dans le fichier [model_observation.py](model_observation.py)

In [1]:
from model_observation import ObserveModel

observer = ObserveModel() # Par default, model_name="Qwen/Qwen2.5-1.5B-Instruct"
input_text = "Quel est le nom de la capitale de la France ?" # L'input utilisateur, içi le modèle va compléter la phrase

input_ids = observer.observe_tokenization(input_text)

S:\IDEProjects\NLP_project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Aucun token de connexion à HuggingFace n'a été fournit / trouvé. Le téléchargement d'un nouveau modèle peut être plus lent ou bloqué en raison des quotas.


Loading weights: 100%|██████████| 338/338 [00:02<00:00, 144.35it/s]


--- Tokenisation ---
Texte d'origine : 'Quel est le nom de la capitale de la France ?'
Input IDs : [2183, 301, 1788, 512, 9662, 409, 1187, 60410, 1574, 409, 1187, 9625, 937]
Tokens décodés un par un : ['Qu', 'el', ' est', ' le', ' nom', ' de', ' la', ' capit', 'ale', ' de', ' la', ' France', ' ?']


# 1. La tokenisation

## Sortie obtenue lors des tests
```
--- Tokenisation ---

Texte d'origine : 'Quel est le nom de la capitale de la France ?'

Input IDs : [2183, 301, 1788, 512, 9662, 409, 1187, 60410, 1574, 409, 1187, 9625, 937] # ID des tokens identifiés

Tokens décodés un par un : ['Qu', 'el', ' est', ' le', ' nom', ' de', ' la', ' capit', 'ale', ' de', ' la', ' France', ' ?']
```

En cours, nous avons utilisé un cas simple de tokenisation : le bag of words. Des mots son égale à un token. Dans les LLM plus complexes et massifs, les tokens sont des parties de mots, des sous mots ou des symboles.

La tokenisation de l'input "Quel est le nom de la capitale de la France ?" est généré par Qwen2.5 en suivant son vocabulaire, les IDS sont quant à eux déterminé via la matrice embedding.
Par définition : la matrice embedding est un dictionnaire de vecteur dans lequel se trouve tous les mots du vocabulaire et leurs vecteurs associés. Celle-ci est aléatoire avant l'entrainement du modèle.

## Comment construire le vocabulaire

L'algorithme utilisé par Qwen2.5 et par beaucoup d'autres modèles de langages s'appelle le **Byte Pair Encoding (BPE)**. Il se décompose en ces étapes :
1. le corpus est donnée en entrée
2. toutes les lettres (et fin de mots) sont considéré comme des tokens
Exemple : Token et Tokens → {t,o,k,e,n,s,</w>, e, t,</w>, t,o,k,e,n,s,</w>} avec </w> le symbole de fin de mot
3. L'algorithme crée des pairs tout le long de la liste : to, ok, ke… s</w> et compte le nombre de pairs similiares. Les plus présentes seront alors fusionées.
Exemple : dans notre exemple, ce sera alors "to" qui sera fusionnée, ce qui donne la liste {to,k,e,n,s,</w>, e, t,</w>, to,k,e,n,s,</w>}
4. Répété l'étape 3 jusqu'à ce que le set (la liste des éléments distincts) soit égale à la taille du vocabulaire recherché

![BPE exemple](../images/bpe_exemple.jpg)

## L'embedding

La taille du vocabulaire de Qwen2.5 est de 151 936 tokens différents. Les Inputs IDs montrés sont les positions des colonnes des tokens décryptés. Içi, l'embedding permet de convertir du texte et des mots en nombres (en vecteurs).

Ainsi, à chacun de ces tokens sera attribué un vecteur différent, reconaissable via son ID.

In [2]:
observer.get_emb_token("Bonjour")

Forme du vecteur : torch.Size([1, 1536])
Vecteur : tensor([[ 0.0227,  0.0459,  0.0059,  ...,  0.0034,  0.0197, -0.0172]],
       dtype=torch.bfloat16)


## Sortie attendu

```
Forme du vecteur : torch.Size([1, 1536])
Vecteur : tensor([[ 0.0227,  0.0459,  0.0059,  ...,  0.0034,  0.0197, -0.0172]],
       dtype=torch.bfloat16)
```

On observe que la taille des vecteurs d'embedding pour le modèle qwen2.5 est de **1536**.

In [3]:
next_token_logits = observer.observe_logits(input_ids)


--- Logits ---
Forme (shape) des logits : torch.Size([1, 13, 151936])


# 2. La forme des Logits

## Sortie obtenue lors des tests
```
--- Logits ---
Forme (shape) des logits : torch.Size([1, 13, 151936])
```

## Analyse de la forme du logits

Ces logits correspondent à la transformation des vecteurs d'embedding via la pipeline suivante  (sans les dimensions associées):
Vecteurs d'embedding des tokens → Transformer n°i { **Attention Causal** (Matrice d'attention) → **MLP** (Multi Layer perceptron,
couche de linéarité et non linéarité)} → **Unembedding** → Matrice des Logits.

Dans les Transformers, les Logits suivent la forme (B,T,C) avec :
- Dimension B : correspond au nombre de batchs en entrée du modèle. Içi 1 pour un seul texte (un seul prompt utilisateur)
- Dimension T : correspond à la variable Time ou "Sequence length", c'est-à-dire la taille du résultat de la tokénisation précédente (voir tokénisation produite plus haut : la liste est bien de taille 13)
- Dimension C : correspond à la variable Channels. à cette étape, cette variable correspond donc à la taille du vocabulaire du modèle (pour Qwen1.5B, c'est ~150000 mots environs).

Ainsi, on comprend que cette matrice (Logits) retournée correspond à la probabilité de prédiction de chaque mot pour le suivant.
Mais la taille est à nuancer : en effet, cette taille est unique au premier token généré par le modèle. En ce qui concerne le second, troisième et n-ème : le logits sera de taille **(1,1,151936)**.

Avec la méthode KV-caching, les modèles ne vont par recalculé des logits de taille (1,13+n,151936) pour tout n tokens dans la réponse, mais va simplement calculer la suite. En effet, cette première matrice contient déjà tous les calculs d'attention nécéssaire pour continuer la génération, il est alors inutile de les recalculer à chaque nouveau token généré ; le modèle ne prend en entrée que le dernier token généré pour continuer (calcule de l'attention).

In [4]:
observer.observe_softmax(next_token_logits)


--- Distribution Softmax ---
Top 1 : ' -' (ID: 481) - Probabilité : 0.2734
Top 2 : ' La' (ID: 4929) - Probabilité : 0.1885
Top 3 : ' Qu' (ID: 3406) - Probabilité : 0.0693
Top 4 : ' Paris' (ID: 12095) - Probabilité : 0.0610
Top 5 : ' (' (ID: 320) - Probabilité : 0.0255


tensor([3.5614e-06, 1.5348e-06, 2.4959e-07,  ..., 2.4374e-10, 2.4374e-10,
        2.4374e-10], dtype=torch.bfloat16)

# 3. Distribution du softmax


## résultat obtenue lors des tests
```
--- Distribution Softmax ---
Top 1 : ' -' (ID: 481) - Probabilité : 0.2734
Top 2 : ' La' (ID: 4929) - Probabilité : 0.1885
Top 3 : ' Qu' (ID: 3406) - Probabilité : 0.0693
Top 4 : ' Paris' (ID: 12095) - Probabilité : 0.0610
Top 5 : ' (' (ID: 320) - Probabilité : 0.0255
tensor([3.5614e-06, 1.5348e-06, 2.4959e-07,  ..., 2.4374e-10, 2.4374e-10,
        2.4374e-10], dtype=torch.bfloat16) # Vrai tenseur
```

## Analyse de la distribution

En utilisant le dernier vecteur généré dans les Logits, celui qui correspond au token n+1 futur, on peut en déduire ces probabilités en le passant dans la fonction softmax qui va donner une probabilité entre 0 et 1 pour chaque token du vocabulaire, içi ~150000 tokens environs.
Pour que ce soit plus visuel et compréhensible, le classement des tokens les plus probables a été décodé à l'avance. En réalité et à cette étape : nous travaillons toujours sur le tenseur marqué dans l'extrait de terminal ci-dessus. La fonction softmax donne les probabilités, mais pas les tokens à choisir, d'où le classement.

On observe par ailleurs que les tokens les plus probables ne sont pas forcéments ceux attendus. À la question "Quel est le nom de la capitale de la France ?", le token le plus probable est " -" soit le début d'une liste. On voit que "Paris" arrive à la 4e place, ce qui signifie que le modèle a bien compris la question.

Le problème vient du prompt en lui-même : les modèles "instruct" ne sont pas que des super autocomplete, ils ont était entrainer avec un **marquage par balise**. Il faut ainsi formater la question comme ceci :
```

<|im_start|>user
Quelle est la capitale de la France ?<|im_end|>
<|im_start|>assistant
```

Avec *<|im_start|>* et *<|im_end|>* deux **tokens entiers**. Avec ce formatage, on incite le LLM à parler comme un assistant.
Cela ne change cependant pas la suite du déroulé, on va ignorer cette incohérence pour l'analyse du modèle.

## Le chat template

Pour compléter cette dernière partie, on peut utiliser le chat template intégré à chaque LLM, ce qui nous permet de découvrir le prompt système et la manière d'encoder les chats pour les LLM.
Exemple avec Qwen2.5:

In [5]:
message_to_template = [
    {"role": "user", "content": "Quelle est la capitale de la France ?"}
]

# Formate le texte correctement pour le modèle Instruct
print(f"Message formaté pour le modèle {observer.model_name} :\n {observer.tokenizer.apply_chat_template(message_to_template, tokenize=False, add_generation_prompt=True)}")

Message formaté pour le modèle Qwen/Qwen2.5-1.5B-Instruct :
 <|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
Quelle est la capitale de la France ?<|im_end|>
<|im_start|>assistant



In [6]:
observer.observe_sampling_strategies(next_token_logits)
observer.observe_top_p(next_token_logits)


--- Stratégies de sélection ---
Argmax -> ' -'


TypeError: 'NoneType' object is not iterable

# 4. Effets de l'argmax, top k et top p
## Extrait lors des tests

```
--- Stratégies de sélection ---
Argmax -> ' -'
Température 0.2 -> Top 1 proba: 0.8789 (' -')
Température 0.7 -> Top 1 proba: 0.4609 (' -')
Température 2.0 -> Top 1 proba: 0.0211 (' -')
Échantillonnage Top-K (K=10) -> Token tiré : ' Qu'

--- Extrait des mots retenus (Top-P = 0.9) ---
Nombre total de mots conservés : 66 / 151936
Les 5 premiers mots du noyau :
  1. ' -' (ID: 481) -> Proba réajustée : 0.3047
  2. ' La' (ID: 4929) -> Proba réajustée : 0.2090
  3. ' Qu' (ID: 3406) -> Proba réajustée : 0.0767
  4. ' Paris' (ID: 12095) -> Proba réajustée : 0.0679
  5. ' (' (ID: 320) -> Proba réajustée : 0.0283

--- Échantillonnage Top-P (P=0.9) ---
Nombre de tokens conservés dans le 'noyau' : 66 / 151936
Token tiré : ' Paris'
```

## Analyse des stratégies de séléction
Pour appliquer les différentes stratégies de séléctions, il faut reprendre le classement précédent :
```
Top 1 : ' -' (ID: 481) - Probabilité : 0.2734
Top 2 : ' La' (ID: 4929) - Probabilité : 0.1885
Top 3 : ' Qu' (ID: 3406) - Probabilité : 0.0693
Top 4 : ' Paris' (ID: 12095) - Probabilité : 0.0610
Top 5 : ' (' (ID: 320) - Probabilité : 0.0255
Top 6 : ...
```

### Argmax
L'**argmax** sélectionne le token le plus probable. Içi, ce sera donc le premier du classement : " -".

### La température

La **température** est un paramètre permettant de lisser ou non les probabilités, cela va donc affecter la créativité du modèle. On voit dans l'exemple plus haut trois températures différentes pour la sélection du token (via l'argmax).
Sur cet exemple, la température n'a aucune incidence sur le token sélectionner, car sa probabilité de base est déjà la plus haute. En réalité, les effets de la température deviennent visible en la combinant avec d'autres stratégies de sélection : Top P et Top K
### Le Top K

Le **top K** consiste à sélectionner les N premiers tokens (içi les 10 premiers) et mettre la probabilité de tous les autres tokens a moins l'infini (-inf) avant de à nouveau appliquer la fonction SoftMax dessus. Ainsi, le nouveau vecteur de probabilité du prochain token aura des probabilités plus hautes pour les tokens gardés. Cette méthode permet de restreindre les possibles prochains tokens tout en excluant la majeure partie du vocabulaire.

### Le Top P

Le principe du **Top p** est de sélectionner les P premiers tokens dont la somme des probabilités dépassent un seuil C.

## Le pipeline décisionnel

Après le passage des logits dans le MLP, on utilise un pipeline pour déterminer quel sera le prochain token à générer.
La voici :
Logits (~150000 dim) → Application de la température (~150000 dim) → Top-K (~150000 dim) → Top-P (~150000 dim) → Softmax (~150000 dim, la distribution est de nouveau égale à 1) → Choix aléatoire parmi les tokens non nuls OU Argmax.

La fonction ci-dessous permet de tester le pipeline complet, avec chaque paramètre modifiable (Température, Top K, Top P, sélection).

In [ ]:

message_to_template = [
    {"role": "user", "content": "Répond en un seul mot. Quelle est la capitale de la France ?"}
]

# Run du pipeline complet, de la tokenisation à la stratégie complète de décision, avec un message formaté pour le modèle Instruct
observer.run_pipeline(observer.tokenizer.apply_chat_template(message_to_template, tokenize=False, add_generation_prompt=True), temperature=0.9, top_k=50, top_p=0.9, use_argmax=True)